In [8]:
import pandas as pd
from pathlib import Path
import json
from tqdm import tqdm
import pickle

In [9]:
filepath = Path("Procedure-Protein-Mapping/Output-Data/test_with_predicted_proteins.csv")
data = pd.read_csv(filepath)

In [ ]:
from reasoner.deductive_reasoner import DeductiveReasoner
from TruthValue import TruthValue

In [11]:
reasoner = DeductiveReasoner()

In [12]:
top_k = 5

In [ ]:
results = []
for index, row in tqdm(data.iterrows(), total=len(data)):
    proteins = row['predicted_proteins']
    proteins = json.loads(proteins.replace("'", "\""))
    proteins = [(p, TruthValue(1.0, 0.9)) for p in proteins if reasoner.valid_protein(p)]
    label = row['diagnoses']
    label = json.loads(label.replace("'", "\""))
    result = reasoner.deductive_reasoning(proteins)
    result = sorted(result, key=lambda x: x[1].e, reverse=True)
    result = result[:top_k]
    results.append((index, result, label))

100%|██████████| 1332/1332 [00:16<00:00, 81.11it/s]


In [147]:
overlapping_rates = []
for result in results:
    pred = set(r[0] for r in result[1])
    label = set(result[2])
    overlapping_rate = len(pred&label)/len(pred|label)
    overlapping_rates.append((overlapping_rate, result))

In [148]:
overlapping_rate, (idx, pred, label) = max(overlapping_rates, key=lambda x: x[0])

### Sample

In [13]:
idx = 675

In [21]:
sample = data.iloc[idx]
procedures = sample["procedures"]
predicted_proteins = json.loads(sample["predicted_proteins"].replace("'", '"'))
proteins = [(p, TruthValue(1.0, 0.9)) for p in predicted_proteins if reasoner.valid_protein(p)]
label = json.loads(sample["diagnoses"].replace("'", '"'))

In [25]:
result = reasoner.deductive_reasoning(proteins)
pred = result[:top_k]


In [28]:
pred

[('75839', <TruthValue: %1.00;0.90% (k=1)>),
 ('7560', <TruthValue: %1.00;0.90% (k=1)>),
 ('2530', <TruthValue: %1.00;0.90% (k=1)>),
 ('2727', <TruthValue: %1.00;0.90% (k=1)>),
 ('75989', <TruthValue: %1.00;0.90% (k=1)>)]

In [29]:
label

['4271',
 '40391',
 '42731',
 '4254',
 '4240',
 '41401',
 '41042',
 'V1251',
 '2859',
 'V1271',
 '3051']

In [30]:
procedures

"['3794', '3723', '8856', '9671', '3995', '9904']"

In [31]:
predicted_proteins

['PROTEIN:21977',
 'PROTEIN:4906',
 'PROTEIN:3775',
 'PROTEIN:4383',
 'PROTEIN:6119',
 'PROTEIN:77',
 'PROTEIN:9683',
 'PROTEIN:1346',
 'PROTEIN:17994',
 'PROTEIN:1530',
 'PROTEIN:12505',
 'PROTEIN:19793',
 'PROTEIN:4417',
 'PROTEIN:6489',
 'PROTEIN:11974',
 'PROTEIN:20115',
 'PROTEIN:4358',
 'PROTEIN:21279',
 'PROTEIN:2706',
 'PROTEIN:16610',
 'PROTEIN:831',
 'PROTEIN:1479',
 'PROTEIN:1426',
 'PROTEIN:9979',
 'PROTEIN:10038',
 'PROTEIN:256',
 'PROTEIN:72',
 'PROTEIN:11486',
 'PROTEIN:1113',
 'PROTEIN:19637',
 'PROTEIN:9685',
 'PROTEIN:19635',
 'PROTEIN:13326',
 'PROTEIN:4405',
 'PROTEIN:11410',
 'PROTEIN:8194',
 'PROTEIN:14369',
 'PROTEIN:15431',
 'PROTEIN:6154',
 'PROTEIN:17135',
 'PROTEIN:7849',
 'PROTEIN:987',
 'PROTEIN:4820',
 'PROTEIN:887',
 'PROTEIN:830',
 'PROTEIN:3585',
 'PROTEIN:8782',
 'PROTEIN:10889',
 'PROTEIN:3834',
 'PROTEIN:6885',
 'PROTEIN:21294',
 'PROTEIN:7559',
 'PROTEIN:939',
 'PROTEIN:8146',
 'PROTEIN:6540',
 'PROTEIN:10036',
 'PROTEIN:3385',
 'PROTEIN:2870',
 'PR

In [32]:
sample['patient_id']

172

In [35]:
_, layer2_result = reasoner.deductive_reasoning([(p, TruthValue(1.0, 0.9)) for p in predicted_proteins if reasoner.valid_protein(p)], return_intermediate_results=True)

In [43]:
len(layer2_result)

1512

In [56]:
sorted(layer2_result, key=lambda x: x[1].e, reverse=True)[:3]

[('GENE:32600', <TruthValue: %1.00;0.90% (k=1)>),
 ('GENE:33174', <TruthValue: %1.00;0.90% (k=1)>),
 ('GENE:35094', <TruthValue: %1.00;0.90% (k=1)>)]

In [1]:
from icd9_to_text import get_description

In [6]:
print(get_description('75839'))

Autosomal deletions NEC
